In [ ]:
### READ ME:
## Full code takes 20 seconds to run with given parameters size, d in main( size , d )

In [2]:
import numpy as np
import matplotlib.pyplot as plt

def initialize(size):
    x = np.zeros(size)
    y = np.zeros(size)
    for i in range(size):
        x[i] = -1+ ((i*2)/(size-1)) # position variables translated from indeces to positions
        y[i]=  -1+ ((i*2)/(size-1))
    potential = np.zeros((size,size)) #Zero potential inside
    for i in range(1,size-1): # Linear gradient between vertical plates
        potential[0][i] = -1+ ((i*2)/(size-1)) 
        potential[-1][i] = -1+ ((i*2)/(size-1)) 
    for i in range(0,size):
        potential[i][0] = -1
        potential[i][-1] = 1
    return x,y, potential

In [3]:
def initialize2(size,d):
    # distance d (integer) 
    # size (integer) must be odd
    x = np.zeros(size)
    y = np.zeros(size)
    for i in range(size):
        x[i] = -1+ ((i*2)/(size-1)) # position variables translated from indeces to positions
        y[i]=  -1+ ((i*2)/(size-1))
    potential = np.zeros((size,size)) #Zero potential inside
    centerx = int(size/2)
    centery = int(size/2)
    cap1 = np.array([[centerx-(d/2),centery],[centerx-(d/2),centery+1],[centerx-(d/2),centery-1],[centerx-(d/2),centery+2],[centerx-(d/2),centery-2],[centerx-(d/2),centery+3],[centerx-(d/2),centery-3],[centerx-(d/2),centery+4],[centerx-(d/2),centery-4],[centerx-(d/2),centery+5],[centerx-(d/2),centery-5]]).astype(int)
    cap2 = np.array([[centerx+(d/2),centery],[centerx+(d/2),centery+1],[centerx+(d/2),centery-1],[centerx+(d/2),centery+2],[centerx+(d/2),centery-2],[centerx+(d/2),centery+3],[centerx+(d/2),centery-3],[centerx+(d/2),centery+4],[centerx+(d/2),centery-4],[centerx+(d/2),centery+5],[centerx+(d/2),centery-5]]).astype(int)
    for i in range(0,size):
        potential[i][0] = 0
        potential[i][-1] = 0
        potential[0][i] = 0
        potential[-1][i]= 0
    for i,j in cap1:
        potential[j][i]=-5
    for i,j in cap2:
        potential[j,i]=5.01
    return x,y, potential,cap1,cap2
#x,y,potential, cap1,cap2 = initialize2(7,2)
#print(potential)


In [4]:
def update_pot(pot,size,deltaV):
    pot_new = np.zeros((size,size))
    for i in range(1,size-1):
        for j in range(1,size-1):
            pot_new[j][i] = pot[j][i]
    for i in range(1,size-1):
        for j in range(1,size-1):
            pot_new[j][i] = (0.25)*(pot[j+1][i] + pot[j-1][i]+pot[j][i-1]+pot[j][i+1])
            deltaV += (pot_new[j][i]-pot[j][i])
    for i in range(1,size-1):
        for j in range(1,size-1):
            pot[j][i] = pot_new[j][i]
    return pot, deltaV

In [5]:
def update_pot2(pot,size,deltaV,cap1,cap2):
    pot_new = np.zeros((size,size))
    for i in range(1,size-1):
        for j in range(1,size-1):
            pot_new[j][i] = pot[j][i]
    for i in range(1,size-1):
        for j in range(1,size-1):
            if np.any(np.all(cap1 == [i, j], axis=1)):
                pot_new[j][i] = -5
            elif np.any(np.all(cap2 == [i, j], axis=1)):
                pot_new[j][i] = 5.01
            else:
                pot_new[j][i] = (0.25)*(pot[j+1][i] + pot[j-1][i]+pot[j][i-1]+pot[j][i+1])
                deltaV += (pot_new[j][i]-pot[j][i])
    for i in range(1,size-1):
        for j in range(1,size-1):
            pot[j][i] = pot_new[j][i]
    return pot, deltaV

In [6]:
def relaxxxxx(pot,size):
    deltaV = 1
    kiter = 0
    while (deltaV > 0.02 and kiter < 90):
        pot, deltaV = update_pot(pot,size,deltaV)
        kiter += 1
    return pot
def relaxxxxx2(pot,size,cap1,cap2):
    deltaV = 1
    kiter = 0
    while (deltaV > 0.2 and kiter < 90):
        pot, deltaV = update_pot2(pot,size,deltaV,cap1,cap2)
        kiter += 1
    #print(kiter)
    return pot

In [7]:
def calcE(size,x,y,pot,cap1,cap2):
    Ex = np.zeros((size,size))
    Ey = np.zeros((size,size))
    for i in range(1,size-1):
        for j in range(1,size-1):
            if np.any(np.all(cap1 == [i, j], axis=1)) or np.any(np.all(cap2 == [i, j], axis=1)):
                Ex[j][i] = 0
                Ey[j][i] = 0
            else:
                Ex[j][i] = -(pot[j,i+1]-pot[j,i-1])/(2*(x[i+1]-x[i]))
                Ey[j][i] = -(pot[j+1,i]-pot[j-1,i])/(2*(y[j+1]-y[j]))
    return Ex,Ey

In [8]:
def monitorSlice(size,E):
    center_vert = []
    center_horiz = []
    fringe_horiz = []
    for i in range(1,size-1):
        center_vert.append(E[i][int(size/2)])
        center_horiz.append(E[int(size/2)][i])
        fringe_horiz.append(E[int(size/2+6)][i])
    return center_vert, center_horiz,fringe_horiz

In [9]:
def monitorPoint(size,E):
    pointA = E[int(size/2)][int(size/2)]
    pointB = E[int(size/2)+5][int(size/2)]
    pointC = E[int(size/2)+7][int(size/2)]
    return pointA, pointB, pointC

In [10]:
def norm(Ex,Ey):
    try:
        norm = Ex / np.sqrt(Ex**2+Ey**2)
        return norm
    except RuntimeWarning:
        return 

In [ ]:
def main(size,d):
    x,y,pot,cap1,cap2 = initialize2(size,d)
    pot = relaxxxxx2(pot,size,cap1,cap2)
    Ex,Ey = calcE(size,x,y,pot,cap1,cap2)
    Ex_norm = Ex / np.sqrt(Ex**2+Ey**2)
    Ey_norm = Ey / np.sqrt(Ex**2+Ey**2)
    Ex_norm[np.isnan(Ex_norm)]=0
    Ey_norm[np.isnan(Ey_norm)]=0
    
    E = np.hypot(Ex, Ey)

    ver,hor,fri = monitorSlice(size,E)
    

    #print(np.round(pot,2))
    plt.figure(figsize=(6,4))
    Vplot = plt.contour(x,y,pot,6,colors="Black")
    plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 12
    })
    plt.clabel(Vplot, inline=True,fontsize=8)
    plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 12
    })
    plt.xlabel(r"$x$-Position [m]")
    plt.ylabel(r"$y$-Position [m]")
    plt.show()

    plt.quiver(x,y,Ex_norm,Ey_norm,E, pivot='mid')
    for i in range(1,size-1):
        for j in range(1,size-1):
            if np.any(np.all(cap1 == [i, j], axis=1)) or np.any(np.all(cap2 == [i, j], axis=1)):
                plt.scatter(x[i],y[j],marker='x',color='red')
    plt.scatter(x[int(size/2)],y[int(size/2)],marker='$A$',color='black',s = 115)
    plt.scatter(x[int(size/2)],y[int(size/2+5)],marker='$B$',color='black',s = 115)
    plt.scatter(x[int(size/2)],y[int(size/2+7)],marker='$C$',color='black',s = 115)
    plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 12
})
    plt.xlabel(r"$x$-Position [m]")
    plt.ylabel(r"$y$-Position [m]")
    plt.colorbar()
    plt.show()

    plt.quiver(x,y,np.zeros((size,size)),np.zeros((size,size)),E, pivot='mid')
    for i in range(1,size-1):
        for j in range(1,size-1):
            if np.any(np.all(cap1 == [i, j], axis=1)) or np.any(np.all(cap2 == [i, j], axis=1)):
                plt.scatter(x[i],y[j],marker='x',color='red',s=105)
    plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 12
    })
    plt.xlabel(r"$x$-Position [m]")
    plt.ylabel(r"$y$-Position [m]")
    plt.axvline(x=x[int(size/2)],color='tab:blue',ls=':',label = r'$x=0$ axis',lw=5)
    plt.axhline(y=y[int(size/2)],color='tab:orange',ls=':',label = r'$y=0$ axis',lw=5)
    plt.axhline(y=y[int(size/2)+6],color='tab:green',ls=':',label = r'$y=6$ axis',lw=5)
    plt.legend()
    #plt.xticks(x)
    plt.show()


    plt.figure(figsize=(6,4))
    plt.scatter(y[1:-1],ver,label = r'$|E|$, $x=0$',marker="x",s=50)
    plt.scatter(x[1:-1],hor,label=r"$|E|$, $y=0$",marker="P",s=50)
    plt.scatter(x[1:-1],fri,label=r"$|E|$, $y=6$",marker="h",s=50)
    plt.xlabel(r"$x$ and $y$ Position [m]")
    plt.ylabel(r"$|~E~|~~[V/m]$")
    done = False
    for i in range(1,size-1):
        for j in range(1,size-1):
            if np.any(np.all(cap1 == [i, j], axis=1)) or np.any(np.all(cap2 == [i, j], axis=1)):
                plt.axvline(x=x[i],color='tab:orange',ls='--')
                plt.axvline(x=y[j],color='tab:blue',ls='--')
                done = True
                print("Left Plate X Position:")
                left = x[i]
                print(left)
                break
        if done:
            break
    done = False
    for i in range(size-1,1,-1):
        for j in range(size-1,1,-1):
            if np.any(np.all(cap1 == [i, j], axis=1)) or np.any(np.all(cap2 == [i, j], axis=1)):
                plt.axvline(x=x[i],color='tab:orange',ls='--',label = r'Cap. $x$-Edge')
                plt.axvline(x=y[j],color='tab:blue',ls='--',label = r'Cap. $y$-Edge')
                done = True
                print("Right Plate X Position:")
                right = x[i]
                print(x[i])
                break
        if done:
            break
    plt.xlim(-1,1.25)
    plt.legend()
    plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 11
})
    print("Plate Separation:")
    print(right-left)
    plt.show()
    d_list = np.arange(0,16,2)
    A_list = []
    B_list = []
    C_list = []
    for d in d_list:
        x,y,pot,cap1,cap2 = initialize2(size,d)
        pot = relaxxxxx2(pot,size,cap1,cap2)
        Ex,Ey = calcE(size,x,y,pot,cap1,cap2)
        Ex_norm = Ex / np.sqrt(Ex**2+Ey**2)
        Ey_norm = Ey / np.sqrt(Ex**2+Ey**2)
        Ex_norm[np.isnan(Ex_norm)]=0
        Ey_norm[np.isnan(Ey_norm)]=0
        E = np.hypot(Ex, Ey)
        A,B,C = monitorPoint(size,E)
        A_list.append(A)
        B_list.append(B)
        C_list.append(C)
    pos_list = (d_list*2)/(size-1)
    plt.figure(figsize=(6,4))
    plt.scatter(pos_list,A_list,marker="x",label=f"$|E|$, Point A (Middle)",s=50)
    plt.scatter(pos_list,B_list,marker="P",label=f"$|E|$, Point B (Edge)",s=50)
    plt.scatter(pos_list,C_list,marker="H",label=f"$|E|$, Point C (Fringe)",s=50)
    print(C_list)
    plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 12
    })
    plt.xlabel(r"Plate Separation $d$ [m]")
    plt.ylabel(r"$|~E~|~~[V/m]$")
    plt.legend()
    plt.xticks(pos_list)
    plt.show()

main(25,4)